# Análise de Dados de ISTs com Apache Spark

Etapa de Big Data do pipeline. Este notebook é executado automaticamente pelo container `bigdata`
(via `jupyter nbconvert --execute`) e consome o CSV tratado pela etapa em R.

A lógica de cada etapa vive no pacote [`ist_bigdata`](ist_bigdata/); aqui ficam apenas a orquestração
e a interpretação dos resultados. Todos os gráficos e o mapa são salvos em `output/`.

## 1. Inicialização do Spark e carregamento dos dados

O Spark foi escolhido por sua capacidade de processamento distribuído, essencial para a análise de grandes
volumes de dados. A `SparkSession` é a porta de entrada para ler dados, processar, aplicar transformações
e treinar modelos. O CSV é lido com `header=True` e `inferSchema=True`, para que o Spark identifique
automaticamente os nomes e os tipos das colunas.

In [ ]:
from ist_bigdata.sessao import iniciar_spark, carregar_dados

spark = iniciar_spark()
spark_df = carregar_dados(spark)

## 2. Pré-processamento

O pré-processamento garante a qualidade dos dados antes da modelagem:

- **Variáveis-alvo** — `tem_ist` (1 se a doença é uma IST) e `curavel` (1 se a IST é curável);
- **StringIndexer** — transforma categorias em índices (`handleInvalid='keep'` evita erros com valores não vistos);
- **OneHotEncoder** — cria vetores binários, evitando que o modelo interprete uma ordem inexistente entre categorias;
- **VectorAssembler** — reúne as variáveis numéricas e categóricas codificadas na coluna `features`.

As features são **idade, renda, gênero, localidade e nível educacional**. A coluna `doenca` fica de fora de propósito:
como `tem_ist` é derivada dela, usá-la como entrada entregaria a resposta ao modelo (*vazamento de dados*).

In [ ]:
from ist_bigdata import preprocessamento

spark_df = preprocessamento.adicionar_variaveis_alvo(spark_df)
encoded_df = preprocessamento.codificar_features(spark_df)

## 3. Processamento de Linguagem Natural (PLN)

A coluna `doenca` é convertida em uma representação numérica via **TF-IDF**
(Tokenizer → StopWordsRemover → HashingTF → IDF), gerando a coluna `textFeatures`.

In [ ]:
from ist_bigdata import pln

tfidf_data = pln.aplicar_tfidf(spark_df)

## 4. Data Warehouse e OLAP

Modelo dimensional (esquema estrela) que permite "fatiar" os dados para responder a diferentes perguntas:

| Tabela | Conteúdo |
|---|---|
| `fato_casos` | Registros de teste: idade, renda, doença, localidade e data |
| `dim_tempo` | Data do teste quebrada em ano, mês e dia |
| `dim_localidade` | Distribuição geográfica dos casos |
| `dim_doenca` | Tipo de doença e se é curável |
| `dim_escolaridade` | Impacto do nível educacional |
| `dim_genero` | Recortes demográficos por gênero |

As tabelas são registradas como views temporárias e consultadas com Spark SQL.

In [ ]:
from ist_bigdata import warehouse, olap

tabelas = warehouse.construir_modelo_dimensional(spark_df)
warehouse.registrar_views(tabelas)

olap.executar_todas(spark)

## 5. Análises no paradigma Hadoop MapReduce

Os dados são convertidos em RDD e processados no estilo MapReduce:

- **Média de renda por tipo de IST** — *Map* `(doenca, (renda, 1))` → *Reduce* soma rendas e contagens → *MapValues* calcula a média;
- **Distribuição por faixa etária** — *Map* `(faixa_etaria, 1)` → *Reduce* soma as ocorrências
  (faixas: 0-17, 18-24, 25-34, 35-44, 45-59 e 60+).

In [ ]:
from ist_bigdata import mapreduce

media_renda = mapreduce.media_renda_por_ist(spark_df)
resultado_faixa = mapreduce.distribuicao_faixa_etaria(spark_df)

## 6. Análise temporal

Contagem de registros por ano de teste, útil para identificar tendências, picos ou quedas
e apoiar a formulação de políticas públicas e estratégias de prevenção.

In [ ]:
from ist_bigdata import temporal

spark_df = temporal.adicionar_ano_teste(spark_df)
casos_por_ano_pd = temporal.casos_por_ano(spark_df)

## 7. Visualização geográfica

Mapa interativo (Folium) em que cada marcador representa uma cidade, com tamanho proporcional à quantidade
de casos. O resultado é salvo em `output/mapa_casos_localidade.html`.

In [ ]:
from ist_bigdata import geo

mapa = geo.mapa_casos_por_localidade(spark_df)

## 8. Modelos de classificação

Objetivo: prever se um paciente tem IST a partir de suas características demográficas e socioeconômicas.
Os dados são divididos em treino (70%) e teste (30%) e três modelos são comparados:

1. **Regressão Logística** — modelo linear, simples e eficiente para problemas binários;
2. **Random Forest** — ensemble de árvores de decisão, robusto a overfitting e capaz de capturar relações não lineares;
3. **Naive Bayes** — baseado no teorema de Bayes com suposição de independência entre as variáveis.

In [ ]:
from ist_bigdata import classificacao

train_data, test_data = classificacao.dividir_treino_teste(encoded_df)
predicoes = classificacao.treinar_e_prever(classificacao.criar_modelos(), train_data, test_data)

### 8.1 Comparação entre os modelos

A acurácia de cada modelo é comparada com um **baseline** que sempre prevê a classe majoritária. Como as classes são
desbalanceadas, um modelo só agrega valor se superar esse piso. O melhor modelo é escolhido a partir dos resultados.

In [ ]:
acuracias = classificacao.comparar_acuracias(predicoes, test_data)
nome_melhor = classificacao.melhor_modelo(acuracias)
melhores_predicoes = predicoes[nome_melhor]

### 8.2 Avaliação do melhor modelo

Métricas no conjunto de teste, matriz de confusão, curva ROC e relatório por classe. A **AUC** quantifica a capacidade
do modelo de distinguir as classes: valores próximos de 1 indicam ótimo desempenho e 0.5 equivale a um classificador aleatório.

In [ ]:
classificacao.avaliar_metricas(melhores_predicoes)
classificacao.plotar_matriz_confusao(melhores_predicoes)
auc = classificacao.plotar_curva_roc(melhores_predicoes)
classificacao.relatorio_classificacao(melhores_predicoes, nome_melhor)

**Interpretação:** sem acesso à coluna `doenca`, os modelos dependem apenas de sinais demográficos e
socioeconômicos para identificar a presença de IST. Os resultados devem ser lidos em relação ao baseline.

## 9. Clusterização (K-Means)

A clusterização descobre grupos naturais nos dados sem usar a variável-alvo. As features são antes **padronizadas**
(média 0, desvio 1) — sem isso, a renda, medida em reais, dominaria as distâncias. O **método do cotovelo** avalia o
custo (WCSS) para K entre 2 e 7; seguimos com **K = 4** e projetamos os grupos em 2 dimensões com **PCA**.

In [ ]:
from ist_bigdata import clusterizacao

clustering_df = clusterizacao.padronizar(encoded_df.select("features"))

custos = clusterizacao.metodo_cotovelo(clustering_df)
clusters = clusterizacao.agrupar(clustering_df)
clusterizacao.plotar_clusters_pca(clusters)

## Encerramento

In [ ]:
spark.stop()